# 03 — GPT Model Architecture

## Scientific Abstract GPT

This notebook imports and validates the reusable decoder-only GPT architecture
created by `00_Common_GPT_Components.ipynb`.

The model architecture is defined only once in:

`src/gpt_components.py`

This notebook does not redefine the model classes.

## Architecture components

- BPE token embeddings
- Positional embeddings
- Masked self-attention
- Multi-head self-attention
- Feed-forward networks
- Residual connections
- Layer normalization
- Transformer blocks
- GPT next-token prediction head

## Notebook objectives

1. Load the BPE tokenizer and tokenizer configuration.
2. Import the common GPT components.
3. Create the GPT model configuration.
4. Initialize the GPT language model.
5. Inspect model dimensions and parameter count.
6. Validate embeddings, attention, Transformer blocks, and output shapes.
7. Verify causal masking.
8. Test next-token loss calculation.
9. Test untrained model generation.
10. Save the final model configuration for training.


In [1]:
!pip install -q tokenizers

In [2]:
from google.colab import drive

drive.mount(
    "/content/drive"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import sys
import json
import math
import importlib

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [4]:
PROJECT_PATH = (
    "/content/drive/MyDrive/"
    "Scientific-Abstract-GPT"
)

SRC_FOLDER = os.path.join(
    PROJECT_PATH,
    "src"
)

COMMON_MODULE_PATH = os.path.join(
    SRC_FOLDER,
    "gpt_components.py"
)

TOKENIZER_FOLDER = os.path.join(
    PROJECT_PATH,
    "data",
    "bpe_tokenizer"
)

TOKENIZER_PATH = os.path.join(
    TOKENIZER_FOLDER,
    "tokenizer.json"
)

TOKENIZER_CONFIG_PATH = os.path.join(
    TOKENIZER_FOLDER,
    "tokenizer_config.json"
)

MODEL_FOLDER = os.path.join(
    PROJECT_PATH,
    "models"
)

MODEL_CONFIG_PATH = os.path.join(
    MODEL_FOLDER,
    "gpt_model_config.json"
)

MODEL_SUMMARY_PATH = os.path.join(
    MODEL_FOLDER,
    "gpt_model_summary.json"
)

os.makedirs(
    MODEL_FOLDER,
    exist_ok=True
)

print("Project path:")
print(PROJECT_PATH)

print("\nCommon module:")
print(COMMON_MODULE_PATH)

print("\nTokenizer:")
print(TOKENIZER_PATH)

print("\nModel folder:")
print(MODEL_FOLDER)

Project path:
/content/drive/MyDrive/Scientific-Abstract-GPT

Common module:
/content/drive/MyDrive/Scientific-Abstract-GPT/src/gpt_components.py

Tokenizer:
/content/drive/MyDrive/Scientific-Abstract-GPT/data/bpe_tokenizer/tokenizer.json

Model folder:
/content/drive/MyDrive/Scientific-Abstract-GPT/models


In [5]:
required_files = {
    "common GPT module": (
        COMMON_MODULE_PATH
    ),
    "BPE tokenizer": (
        TOKENIZER_PATH
    ),
    "tokenizer configuration": (
        TOKENIZER_CONFIG_PATH
    )
}

missing_files = []

for file_name, file_path in (
    required_files.items()
):

    exists = os.path.exists(
        file_path
    )

    print(
        f"{file_name}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    if not exists:

        missing_files.append(
            file_path
        )

if missing_files:

    raise FileNotFoundError(
        "The following required files "
        "were not found:\n"
        + "\n".join(missing_files)
    )

print(
    "\nAll required files are available."
)

common GPT module: FOUND
BPE tokenizer: FOUND
tokenizer configuration: FOUND

All required files are available.


In [6]:
if PROJECT_PATH not in sys.path:

    sys.path.insert(
        0,
        PROJECT_PATH
    )

print(
    "Project folder added to Python path:"
)

print(
    PROJECT_PATH
)

Project folder added to Python path:
/content/drive/MyDrive/Scientific-Abstract-GPT


Import Common GPT Components

In [7]:
from src.gpt_components import (
    GPTConfig,
    CausalSelfAttentionHead,
    MultiHeadAttention,
    FeedForward,
    TransformerBlock,
    GPTLanguageModel,
    set_seed,
    count_parameters,
    load_tokenizer,
    create_prompt,
    extract_abstract
)

print(
    "Common GPT components imported successfully."
)

Common GPT components imported successfully.


Reload the Common Module During Development

In [8]:
import src.gpt_components as gpt_components

importlib.reload(
    gpt_components
)

from src.gpt_components import (
    GPTConfig,
    CausalSelfAttentionHead,
    MultiHeadAttention,
    FeedForward,
    TransformerBlock,
    GPTLanguageModel,
    set_seed,
    count_parameters,
    load_tokenizer,
    create_prompt,
    extract_abstract
)

print(
    "Common module reloaded successfully."
)

Common module reloaded successfully.


In [9]:
SEED = 42

set_seed(
    seed=SEED
)

print(
    "Random seed:",
    SEED
)

Random seed: 42


In [10]:
device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Selected device:",
    device
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    gpu_memory_gb = (
        torch.cuda.get_device_properties(
            0
        ).total_memory /
        (1024 ** 3)
    )

    print(
        "GPU memory:",
        f"{gpu_memory_gb:.2f} GB"
    )

Selected device: cuda:0
GPU: Tesla T4
GPU memory: 14.56 GB


Load the BPE Tokenizer

In [11]:
tokenizer = load_tokenizer(
    TOKENIZER_PATH
)

vocab_size = (
    tokenizer.get_vocab_size()
)

print(
    "Tokenizer loaded successfully."
)

print(
    "Vocabulary size:",
    f"{vocab_size:,}"
)

Tokenizer loaded successfully.
Vocabulary size: 8,000


In [12]:
with open(
    TOKENIZER_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:

    tokenizer_config = json.load(
        file
    )

print(
    json.dumps(
        tokenizer_config,
        indent=2
    )
)

{
  "tokenizer_type": "ByteLevelBPE",
  "vocab_size_target": 8000,
  "vocab_size_actual": 8000,
  "minimum_frequency": 2,
  "special_tokens": [
    "<PAD>",
    "<UNK>",
    "<TITLE>",
    "<SUBJECT>",
    "<ABSTRACT>",
    "<END>"
  ],
  "special_token_ids": {
    "<PAD>": 0,
    "<UNK>": 1,
    "<TITLE>": 2,
    "<SUBJECT>": 3,
    "<ABSTRACT>": 4,
    "<END>": 5
  },
  "pad_token": "<PAD>",
  "pad_token_id": 0,
  "unk_token": "<UNK>",
  "unk_token_id": 1,
  "title_token": "<TITLE>",
  "title_token_id": 2,
  "subject_token": "<SUBJECT>",
  "subject_token_id": 3,
  "abstract_token": "<ABSTRACT>",
  "abstract_token_id": 4,
  "end_token": "<END>",
  "end_token_id": 5,
  "recommended_block_size": 256,
  "tokenizer_json_path": "/content/drive/MyDrive/Scientific-Abstract-GPT/data/bpe_tokenizer/tokenizer.json",
  "tokenized_dataset_path": "/content/drive/MyDrive/Scientific-Abstract-GPT/data/tokenized_bpe_dataset",
  "token_stream_path": "/content/drive/MyDrive/Scientific-Abstract-GPT/data/t

Verify Vocabulary Size Consistency

In [13]:
saved_vocab_size = (
    tokenizer_config.get(
        "vocab_size_actual"
    )
)

if saved_vocab_size is None:

    saved_vocab_size = (
        tokenizer_config.get(
            "vocab_size"
        )
    )

if saved_vocab_size is None:

    raise KeyError(
        "Vocabulary size was not found in "
        "tokenizer_config.json."
    )

if int(saved_vocab_size) != vocab_size:

    raise ValueError(
        "Tokenizer vocabulary-size mismatch. "
        f"Tokenizer reports {vocab_size}, but "
        f"configuration reports {saved_vocab_size}."
    )

print(
    "Vocabulary-size consistency check passed."
)

Vocabulary-size consistency check passed.


In [14]:
REQUIRED_SPECIAL_TOKENS = [
    "<PAD>",
    "<UNK>",
    "<TITLE>",
    "<SUBJECT>",
    "<ABSTRACT>",
    "<END>"
]

special_token_ids = {}

for token in REQUIRED_SPECIAL_TOKENS:

    token_id = tokenizer.token_to_id(
        token
    )

    special_token_ids[
        token
    ] = token_id

    print(
        f"{token}: {token_id}"
    )

missing_tokens = [
    token
    for token, token_id
    in special_token_ids.items()
    if token_id is None
]

if missing_tokens:

    raise ValueError(
        "The tokenizer is missing the "
        "following special tokens: "
        f"{missing_tokens}"
    )

print(
    "\nAll required special tokens are available."
)

<PAD>: 0
<UNK>: 1
<TITLE>: 2
<SUBJECT>: 3
<ABSTRACT>: 4
<END>: 5

All required special tokens are available.


Define GPT Architecture Configuration

In [15]:
BLOCK_SIZE = 256
N_EMBD = 256
N_HEAD = 8
N_LAYER = 6
DROPOUT = 0.1
USE_BIAS = True

model_config = GPTConfig(
    vocab_size=vocab_size,
    block_size=BLOCK_SIZE,
    n_embd=N_EMBD,
    n_head=N_HEAD,
    n_layer=N_LAYER,
    dropout=DROPOUT,
    bias=USE_BIAS
)

print(model_config)

GPTConfig(vocab_size=8000, block_size=256, n_embd=256, n_head=8, n_layer=6, dropout=0.1, bias=True)


Display Model Configuration

In [16]:
model_configuration_table = pd.DataFrame({
    "Parameter": [
        "Vocabulary size",
        "Context length",
        "Embedding dimension",
        "Attention heads",
        "Dimension per head",
        "Transformer layers",
        "Dropout",
        "Linear-layer bias"
    ],
    "Value": [
        model_config.vocab_size,
        model_config.block_size,
        model_config.n_embd,
        model_config.n_head,
        model_config.head_size,
        model_config.n_layer,
        model_config.dropout,
        model_config.bias
    ]
})

model_configuration_table

,Parameter,Value
0,Vocabulary size,8000
1,Context length,256
2,Embedding dimension,256
3,Attention heads,8
4,Dimension per head,32
5,Transformer layers,6
6,Dropout,0.1
7,Linear-layer bias,True


In [17]:
configuration_checks = {
    "vocabulary_size_positive": (
        model_config.vocab_size > 0
    ),
    "block_size_positive": (
        model_config.block_size > 0
    ),
    "embedding_size_positive": (
        model_config.n_embd > 0
    ),
    "attention_heads_positive": (
        model_config.n_head > 0
    ),
    "embedding_divisible_by_heads": (
        model_config.n_embd
        % model_config.n_head
        == 0
    ),
    "layers_positive": (
        model_config.n_layer > 0
    ),
    "dropout_valid": (
        0 <= model_config.dropout < 1
    )
}

for check_name, result in (
    configuration_checks.items()
):

    print(
        f"{check_name}: "
        f"{'PASSED' if result else 'FAILED'}"
    )

if not all(
    configuration_checks.values()
):

    raise ValueError(
        "Model configuration validation failed."
    )

vocabulary_size_positive: PASSED
block_size_positive: PASSED
embedding_size_positive: PASSED
attention_heads_positive: PASSED
embedding_divisible_by_heads: PASSED
layers_positive: PASSED
dropout_valid: PASSED


Create the GPT Language Model

In [18]:
if "model" in locals() and model is not None:

    del model

if torch.cuda.is_available():

    torch.cuda.empty_cache()

model = GPTLanguageModel(
    model_config
).to(device)

model_device = next(
    model.parameters()
).device

print(
    "GPT model created successfully."
)

print(
    "\nSelected device:",
    device
)

print(
    "Model device:",
    model_device
)

GPT model created successfully.

Selected device: cuda:0
Model device: cuda:0


In [19]:
print(model)

GPTLanguageModel(
  (token_embedding): Embedding(8000, 256)
  (position_embedding): Embedding(256, 256)
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (layer_norm_attention): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (self_attention): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x CausalSelfAttentionHead(
            (key): Linear(in_features=256, out_features=32, bias=True)
            (query): Linear(in_features=256, out_features=32, bias=True)
            (value): Linear(in_features=256, out_features=32, bias=True)
            (attention_dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (output_projection): Linear(in_features=256, out_features=256, bias=True)
        (residual_dropout): Dropout(p=0.1, inplace=False)
      )
      (layer_norm_feed_forward): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (feed_forward): FeedForward(
     

In [20]:
model_device = next(
    model.parameters()
).device

model_on_selected_device = (
    model_device.type
    == device.type
)

if device.type == "cuda":

    selected_device_index = (
        0
        if device.index is None
        else device.index
    )

    model_device_index = (
        0
        if model_device.index is None
        else model_device.index
    )

    model_on_selected_device = (
        model_on_selected_device
        and model_device_index
        == selected_device_index
    )

print(
    "Selected device:",
    device
)

print(
    "Actual model device:",
    model_device
)

print(
    "Model on selected device:",
    model_on_selected_device
)

if not model_on_selected_device:

    raise RuntimeError(
        "The model is not on the selected device."
    )

Selected device: cuda:0
Actual model device: cuda:0
Model on selected device: True


In [21]:
trainable_parameters = (
    count_parameters(
        model,
        trainable_only=True
    )
)

total_parameters = (
    count_parameters(
        model,
        trainable_only=False
    )
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Approximate parameter count:",
    f"{trainable_parameters / 1_000_000:.2f} million"
)

Trainable parameters: 6,852,608
Total parameters: 6,852,608
Approximate parameter count: 6.85 million


Estimate Parameter Memory

This estimate covers model parameters only. Training requires additional memory for gradients, optimizer states, activations, and batches.

In [22]:
bytes_per_float32 = 4

parameter_memory_bytes = (
    total_parameters *
    bytes_per_float32
)

parameter_memory_mb = (
    parameter_memory_bytes /
    (1024 ** 2)
)

print(
    "Approximate FP32 parameter memory:",
    f"{parameter_memory_mb:.2f} MB"
)

Approximate FP32 parameter memory: 26.14 MB


Inspect Token Embedding Layer

The token embedding converts each BPE token ID into a dense vector.

In [23]:
token_embedding_layer = (
    model.token_embedding
)

print(
    token_embedding_layer
)

print(
    "\nEmbedding weight shape:",
    token_embedding_layer.weight.shape
)

Embedding(8000, 256)

Embedding weight shape: torch.Size([8000, 256])


Test Token Embeddings

In [24]:
embedding_test_ids = torch.tensor(
    [
        [
            special_token_ids[
                "<TITLE>"
            ],
            special_token_ids[
                "<SUBJECT>"
            ],
            special_token_ids[
                "<ABSTRACT>"
            ],
            special_token_ids[
                "<END>"
            ]
        ]
    ],
    dtype=torch.long,
    device=device
)

with torch.no_grad():

    token_embedding_output = (
        model.token_embedding(
            embedding_test_ids
        )
    )

print(
    "Input token shape:",
    embedding_test_ids.shape
)

print(
    "Token embedding shape:",
    token_embedding_output.shape
)

Input token shape: torch.Size([1, 4])
Token embedding shape: torch.Size([1, 4, 256])


Inspect Positional Embedding Layer

In [25]:
position_embedding_layer = (
    model.position_embedding
)

print(
    position_embedding_layer
)

print(
    "\nPosition embedding weight shape:",
    position_embedding_layer.weight.shape
)

Embedding(256, 256)

Position embedding weight shape: torch.Size([256, 256])


In [26]:
test_sequence_length = 16

position_ids = torch.arange(
    test_sequence_length,
    dtype=torch.long,
    device=device
)

with torch.no_grad():

    position_embedding_output = (
        model.position_embedding(
            position_ids
        )
    )

print(
    "Position IDs shape:",
    position_ids.shape
)

print(
    "Position embedding shape:",
    position_embedding_output.shape
)

Position IDs shape: torch.Size([16])
Position embedding shape: torch.Size([16, 256])


Test Combined Embeddings

In [27]:
test_input_ids = torch.randint(
    low=0,
    high=model_config.vocab_size,
    size=(
        2,
        test_sequence_length
    ),
    dtype=torch.long,
    device=device
)

with torch.no_grad():

    test_token_embeddings = (
        model.token_embedding(
            test_input_ids
        )
    )

    test_position_embeddings = (
        model.position_embedding(
            position_ids
        )
    )

    combined_embeddings = (
        test_token_embeddings +
        test_position_embeddings
    )

print(
    "Token embeddings:",
    test_token_embeddings.shape
)

print(
    "Position embeddings:",
    test_position_embeddings.shape
)

print(
    "Combined embeddings:",
    combined_embeddings.shape
)

Token embeddings: torch.Size([2, 16, 256])
Position embeddings: torch.Size([16, 256])
Combined embeddings: torch.Size([2, 16, 256])


Create and Test One Attention Head

In [28]:
attention_head = (
    CausalSelfAttentionHead(
        model_config
    )
).to(device)

attention_head.eval()

with torch.no_grad():

    attention_head_output = (
        attention_head(
            combined_embeddings
        )
    )

print(
    "Attention-head input shape:",
    combined_embeddings.shape
)

print(
    "Attention-head output shape:",
    attention_head_output.shape
)

print(
    "Dimension per head:",
    model_config.head_size
)

Attention-head input shape: torch.Size([2, 16, 256])
Attention-head output shape: torch.Size([2, 16, 32])
Dimension per head: 32


Inspect the Causal Attention Mask

In [29]:
causal_mask = (
    attention_head.causal_mask[
        :10,
        :10
    ]
    .detach()
    .cpu()
    .numpy()
    .astype(int)
)

causal_mask_dataframe = pd.DataFrame(
    causal_mask,
    index=[
        f"Query {index}"
        for index in range(10)
    ],
    columns=[
        f"Key {index}"
        for index in range(10)
    ]
)

causal_mask_dataframe

,Key 0,Key 1,Key 2,Key 3,Key 4,Key 5,Key 6,Key 7,Key 8,Key 9
Query 0,1,0,0,0,0,0,0,0,0,0
Query 1,1,1,0,0,0,0,0,0,0,0
Query 2,1,1,1,0,0,0,0,0,0,0
Query 3,1,1,1,1,0,0,0,0,0,0
Query 4,1,1,1,1,1,0,0,0,0,0
Query 5,1,1,1,1,1,1,0,0,0,0
Query 6,1,1,1,1,1,1,1,0,0,0
Query 7,1,1,1,1,1,1,1,1,0,0
Query 8,1,1,1,1,1,1,1,1,1,0
Query 9,1,1,1,1,1,1,1,1,1,1


Validate the Causal Mask

In [30]:
expected_causal_mask = torch.tril(
    torch.ones(
        10,
        10,
        dtype=torch.bool
    )
)

actual_causal_mask = (
    attention_head.causal_mask[
        :10,
        :10
    ].cpu()
)

causal_mask_is_valid = torch.equal(
    actual_causal_mask,
    expected_causal_mask
)

print(
    "Causal mask valid:",
    causal_mask_is_valid
)

if not causal_mask_is_valid:

    raise ValueError(
        "The causal attention mask is incorrect."
    )

Causal mask valid: True


Verify Future Tokens Do Not Affect Earlier Outputs

This is a stronger causal-attention test.

Two inputs are identical at the beginning but different later. The output for the early positions should remain the same.

In [31]:
causal_test_head = (
    CausalSelfAttentionHead(
        model_config
    )
).to(device)

causal_test_head.eval()

causal_input_a = torch.randn(
    1,
    8,
    model_config.n_embd,
    device=device
)

causal_input_b = (
    causal_input_a.clone()
)

# Change only future positions.
causal_input_b[
    :,
    4:,
    :
] = torch.randn_like(
    causal_input_b[
        :,
        4:,
        :
    ]
)

with torch.no_grad():

    causal_output_a = (
        causal_test_head(
            causal_input_a
        )
    )

    causal_output_b = (
        causal_test_head(
            causal_input_b
        )
    )

early_output_difference = (
    causal_output_a[
        :,
        :4,
        :
    ] -
    causal_output_b[
        :,
        :4,
        :
    ]
).abs().max().item()

print(
    "Maximum difference in early outputs:",
    early_output_difference
)

causal_behavior_valid = (
    early_output_difference < 1e-6
)

print(
    "Future-token isolation valid:",
    causal_behavior_valid
)

if not causal_behavior_valid:

    raise ValueError(
        "Future tokens affected earlier "
        "attention outputs."
    )

Maximum difference in early outputs: 0.0
Future-token isolation valid: True


Test Multi-Head Attention

In [32]:
multi_head_attention = (
    MultiHeadAttention(
        model_config
    )
).to(device)

multi_head_attention.eval()

with torch.no_grad():

    multi_head_output = (
        multi_head_attention(
            combined_embeddings
        )
    )

print(
    "Multi-head input shape:",
    combined_embeddings.shape
)

print(
    "Multi-head output shape:",
    multi_head_output.shape
)

print(
    "Number of attention heads:",
    model_config.n_head
)

Multi-head input shape: torch.Size([2, 16, 256])
Multi-head output shape: torch.Size([2, 16, 256])
Number of attention heads: 8


Test Feed-Forward Network

In [33]:
feed_forward = FeedForward(
    model_config
).to(device)

feed_forward.eval()

with torch.no_grad():

    feed_forward_output = (
        feed_forward(
            combined_embeddings
        )
    )

print(
    "Feed-forward input shape:",
    combined_embeddings.shape
)

print(
    "Feed-forward output shape:",
    feed_forward_output.shape
)

Feed-forward input shape: torch.Size([2, 16, 256])
Feed-forward output shape: torch.Size([2, 16, 256])


Inspect Feed-Forward Expansion

In [34]:
first_linear_layer = (
    feed_forward.network[0]
)

second_linear_layer = (
    feed_forward.network[2]
)

print(
    "First linear layer:"
)

print(
    first_linear_layer
)

print(
    "\nSecond linear layer:"
)

print(
    second_linear_layer
)

print(
    "\nExpansion dimension:",
    4 * model_config.n_embd
)

First linear layer:
Linear(in_features=256, out_features=1024, bias=True)

Second linear layer:
Linear(in_features=1024, out_features=256, bias=True)

Expansion dimension: 1024


Test One Transformer Block

In [35]:
transformer_block = (
    TransformerBlock(
        model_config
    )
).to(device)

transformer_block.eval()

with torch.no_grad():

    transformer_block_output = (
        transformer_block(
            combined_embeddings
        )
    )

print(
    "Transformer-block input shape:",
    combined_embeddings.shape
)

print(
    "Transformer-block output shape:",
    transformer_block_output.shape
)

Transformer-block input shape: torch.Size([2, 16, 256])
Transformer-block output shape: torch.Size([2, 16, 256])


Verify Residual Shape Preservation

In [36]:
residual_shape_valid = (
    transformer_block_output.shape
    == combined_embeddings.shape
)

print(
    "Residual shape preserved:",
    residual_shape_valid
)

if not residual_shape_valid:

    raise ValueError(
        "Transformer block did not preserve "
        "the embedding dimensions."
    )

Residual shape preserved: True


In [37]:
sample_title = (
    "Deep Learning for "
    "Medical Image Classification"
)

sample_subject = (
    "Machine Learning"
)

sample_prompt = create_prompt(
    title=sample_title,
    subject=sample_subject
)

print(sample_prompt)

<TITLE> Deep Learning for Medical Image Classification <SUBJECT> Machine Learning <ABSTRACT>


In [38]:
prompt_encoding = tokenizer.encode(
    sample_prompt
)

prompt_token_ids = (
    prompt_encoding.ids
)

prompt_tokens = (
    prompt_encoding.tokens
)

print(
    "Number of prompt tokens:",
    len(prompt_token_ids)
)

print(
    "\nPrompt token IDs:"
)

print(
    prompt_token_ids
)

print(
    "\nPrompt tokens:"
)

print(
    prompt_tokens
)

Number of prompt tokens: 13

Prompt token IDs:
[2, 1290, 468, 324, 5478, 3843, 2706, 226, 3, 536, 468, 226, 4]

Prompt tokens:
['<TITLE>', 'ĠDeep', 'ĠLearning', 'Ġfor', 'ĠMedical', 'ĠImage', 'ĠClassification', 'Ġ', '<SUBJECT>', 'ĠMachine', 'ĠLearning', 'Ġ', '<ABSTRACT>']


In [39]:
decoded_prompt = tokenizer.decode(
    prompt_token_ids,
    skip_special_tokens=False
)

print(
    "Original prompt:"
)

print(
    sample_prompt
)

print(
    "\nDecoded prompt:"
)

print(
    decoded_prompt
)

Original prompt:
<TITLE> Deep Learning for Medical Image Classification <SUBJECT> Machine Learning <ABSTRACT>

Decoded prompt:
<TITLE> Deep Learning for Medical Image Classification <SUBJECT> Machine Learning <ABSTRACT>


Convert Prompt to a Tensor

In [40]:
prompt_tensor = torch.tensor(
    prompt_token_ids,
    dtype=torch.long,
    device=device
).unsqueeze(0)

print(
    "Prompt tensor shape:",
    prompt_tensor.shape
)

print(
    "Prompt tensor device:",
    prompt_tensor.device
)

Prompt tensor shape: torch.Size([1, 13])
Prompt tensor device: cuda:0


Test GPT Forward Pass Without Targets

In [41]:
model.eval()

with torch.no_grad():

    prompt_logits, prompt_loss = model(
        prompt_tensor
    )

print(
    "Prompt input shape:",
    prompt_tensor.shape
)

print(
    "Prompt logits shape:",
    prompt_logits.shape
)

print(
    "Loss:",
    prompt_loss
)

Prompt input shape: torch.Size([1, 13])
Prompt logits shape: torch.Size([1, 13, 8000])
Loss: None


Validate GPT Output Shape

In [42]:
expected_prompt_logits_shape = (
    1,
    len(prompt_token_ids),
    model_config.vocab_size
)

actual_prompt_logits_shape = tuple(
    prompt_logits.shape
)

print(
    "Expected shape:",
    expected_prompt_logits_shape
)

print(
    "Actual shape:",
    actual_prompt_logits_shape
)

if (
    actual_prompt_logits_shape
    != expected_prompt_logits_shape
):

    raise ValueError(
        "GPT output shape is incorrect."
    )

print(
    "GPT output-shape validation passed."
)

Expected shape: (1, 13, 8000)
Actual shape: (1, 13, 8000)
GPT output-shape validation passed.


Create a Next-Token Prediction Batch

In [43]:
BATCH_SIZE = 4
SEQUENCE_LENGTH = 64

training_test_data = torch.randint(
    low=0,
    high=model_config.vocab_size,
    size=(
        BATCH_SIZE,
        SEQUENCE_LENGTH + 1
    ),
    dtype=torch.long,
    device=device
)

input_batch = training_test_data[
    :,
    :-1
]

target_batch = training_test_data[
    :,
    1:
]

print(
    "Input batch shape:",
    input_batch.shape
)

print(
    "Target batch shape:",
    target_batch.shape
)

Input batch shape: torch.Size([4, 64])
Target batch shape: torch.Size([4, 64])


In [44]:
model.train()

training_logits, training_loss = model(
    input_batch,
    target_batch
)

print(
    "Training logits shape:",
    training_logits.shape
)

print(
    "Training loss:",
    training_loss.item()
)

Training logits shape: torch.Size([4, 64, 8000])
Training loss: 9.02022933959961


Validate Loss

In [45]:
loss_is_scalar = (
    training_loss.ndim == 0
)

loss_is_finite = bool(
    torch.isfinite(
        training_loss
    ).item()
)

loss_is_positive = (
    training_loss.item() > 0
)

print(
    "Loss is scalar:",
    loss_is_scalar
)

print(
    "Loss is finite:",
    loss_is_finite
)

print(
    "Loss is positive:",
    loss_is_positive
)

if not (
    loss_is_scalar
    and loss_is_finite
    and loss_is_positive
):

    raise ValueError(
        "The next-token prediction loss is invalid."
    )

Loss is scalar: True
Loss is finite: True
Loss is positive: True


Compare Initial Loss with Random Baseline

For an untrained language model, the expected initial cross-entropy loss is approximately:

In [46]:
expected_random_loss = math.log(
    model_config.vocab_size
)

actual_initial_loss = (
    training_loss.item()
)

print(
    "Expected random baseline loss:",
    f"{expected_random_loss:.4f}"
)

print(
    "Observed initial loss:",
    f"{actual_initial_loss:.4f}"
)

print(
    "Absolute difference:",
    f"{abs(actual_initial_loss - expected_random_loss):.4f}"
)

Expected random baseline loss: 8.9872
Observed initial loss: 9.0202
Absolute difference: 0.0330


Test Backpropagation

In [47]:
model.zero_grad(
    set_to_none=True
)

training_logits, training_loss = model(
    input_batch,
    target_batch
)

training_loss.backward()

parameters_with_gradients = sum(
    1
    for parameter in model.parameters()
    if parameter.grad is not None
)

total_trainable_tensors = sum(
    1
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    "Trainable parameter tensors:",
    total_trainable_tensors
)

print(
    "Parameter tensors with gradients:",
    parameters_with_gradients
)

Trainable parameter tensors: 352
Parameter tensors with gradients: 352


Validate Gradients

In [48]:
gradient_values_are_finite = True

for parameter in model.parameters():

    if parameter.grad is None:
        continue

    if not torch.isfinite(
        parameter.grad
    ).all():

        gradient_values_are_finite = False
        break

print(
    "Gradient values are finite:",
    gradient_values_are_finite
)

if not gradient_values_are_finite:

    raise ValueError(
        "Non-finite gradients were detected."
    )

model.zero_grad(
    set_to_none=True
)

Gradient values are finite: True


Verify Weight Tying

The token embedding matrix and output language-model head should share the same weights.

In [49]:
embedding_weight_pointer = (
    model.token_embedding.weight.data_ptr()
)

output_weight_pointer = (
    model.language_model_head.weight.data_ptr()
)

weights_are_tied = (
    embedding_weight_pointer
    == output_weight_pointer
)

print(
    "Input and output weights are tied:",
    weights_are_tied
)

if not weights_are_tied:

    raise ValueError(
        "Weight tying is not configured correctly."
    )

Input and output weights are tied: True


Test Untrained Token Generation


In [50]:
model.eval()

END_TOKEN_ID = (
    special_token_ids[
        "<END>"
    ]
)

with torch.no_grad():

    generated_ids = model.generate(
        input_ids=prompt_tensor,
        max_new_tokens=30,
        temperature=0.8,
        top_k=40,
        top_p=None,
        repetition_penalty=1.1,
        end_token_id=END_TOKEN_ID
    )

print(
    "Prompt length:",
    prompt_tensor.shape[1]
)

print(
    "Generated sequence length:",
    generated_ids.shape[1]
)

print(
    "\nGenerated token IDs:"
)

print(
    generated_ids[
        0
    ].detach().cpu().tolist()
)

Prompt length: 13
Generated sequence length: 43

Generated token IDs:
[2, 1290, 468, 324, 5478, 3843, 2706, 226, 3, 536, 468, 226, 4, 6774, 4368, 739, 1376, 4815, 7514, 7078, 14, 2504, 2400, 963, 1407, 1614, 3056, 1192, 365, 3470, 3288, 3288, 6513, 4561, 78, 4392, 300, 96, 5322, 7628, 6519, 3747, 3747]


Decode Untrained Generation

In [51]:
untrained_generation = (
    tokenizer.decode(
        generated_ids[
            0
        ].detach().cpu().tolist(),
        skip_special_tokens=False
    )
)

print(
    untrained_generation
)

<TITLE> Deep Learning for Medical Image Classification <SUBJECT> Machine Learning <ABSTRACT> vulnerability Buildingors htt place contributing transparent) sentences fil form adversarialimensionalramsentalem Understandingtextittextit 2020 transformersi Sumel{gebafe overl hypothesis hypothesis


Extract Abstract from Untrained Output

In [52]:
untrained_abstract = extract_abstract(
    untrained_generation
)

print(
    "Extracted abstract:"
)

print(
    untrained_abstract
)

Extracted abstract:
vulnerability Buildingors htt place contributing transparent) sentences fil form adversarialimensionalramsentalem Understandingtextittextit 2020 transformersi Sumel{gebafe overl hypothesis hypothesis


Validate Maximum Context Length

In [53]:
maximum_context_input = torch.randint(
    low=0,
    high=model_config.vocab_size,
    size=(
        1,
        model_config.block_size
    ),
    dtype=torch.long,
    device=device
)

model.eval()

with torch.no_grad():

    maximum_context_logits, _ = model(
        maximum_context_input
    )

print(
    "Maximum context input shape:",
    maximum_context_input.shape
)

print(
    "Maximum context output shape:",
    maximum_context_logits.shape
)

Maximum context input shape: torch.Size([1, 256])
Maximum context output shape: torch.Size([1, 256, 8000])


Verify Oversized Context Is Rejected

In [54]:
oversized_context_input = torch.randint(
    low=0,
    high=model_config.vocab_size,
    size=(
        1,
        model_config.block_size + 1
    ),
    dtype=torch.long,
    device=device
)

oversized_context_rejected = False

try:

    with torch.no_grad():

        model(
            oversized_context_input
        )

except ValueError as error:

    oversized_context_rejected = True

    print(
        "Expected error received:"
    )

    print(error)

print(
    "\nOversized context rejected:",
    oversized_context_rejected
)

if not oversized_context_rejected:

    raise ValueError(
        "The model accepted an input longer "
        "than its configured block size."
    )

Expected error received:
Input sequence length exceeds block_size. Received 257, but block_size is 256.

Oversized context rejected: True


In [55]:
architecture_summary = {
    "model_name": (
        "Scientific Abstract GPT"
    ),
    "model_type": (
        "Decoder-only Transformer"
    ),
    "tokenizer_type": (
        "Byte-Level BPE"
    ),
    "configuration": (
        model_config.to_dict()
    ),
    "derived_values": {
        "head_size": (
            model_config.head_size
        ),
        "feed_forward_hidden_size": (
            4 * model_config.n_embd
        ),
        "trainable_parameters": (
            trainable_parameters
        ),
        "total_parameters": (
            total_parameters
        ),
        "parameters_millions": round(
            trainable_parameters /
            1_000_000,
            4
        ),
        "estimated_fp32_parameter_memory_mb": round(
            parameter_memory_mb,
            2
        )
    },
    "special_token_ids": (
        special_token_ids
    ),
    "components": [
        "Token embedding",
        "Positional embedding",
        "Masked self-attention",
        "Multi-head attention",
        "Feed-forward network",
        "Transformer block",
        "Layer normalization",
        "Residual connection",
        "Language-model output head"
    ],
    "common_module": (
        COMMON_MODULE_PATH
    ),
    "tokenizer_path": (
        TOKENIZER_PATH
    )
}

print(
    json.dumps(
        architecture_summary,
        indent=2
    )
)

{
  "model_name": "Scientific Abstract GPT",
  "model_type": "Decoder-only Transformer",
  "tokenizer_type": "Byte-Level BPE",
  "configuration": {
    "vocab_size": 8000,
    "block_size": 256,
    "n_embd": 256,
    "n_head": 8,
    "n_layer": 6,
    "dropout": 0.1,
    "bias": true
  },
  "derived_values": {
    "head_size": 32,
    "feed_forward_hidden_size": 1024,
    "trainable_parameters": 6852608,
    "total_parameters": 6852608,
    "parameters_millions": 6.8526,
    "estimated_fp32_parameter_memory_mb": 26.14
  },
  "special_token_ids": {
    "<PAD>": 0,
    "<UNK>": 1,
    "<TITLE>": 2,
    "<SUBJECT>": 3,
    "<ABSTRACT>": 4,
    "<END>": 5
  },
  "components": [
    "Token embedding",
    "Positional embedding",
    "Masked self-attention",
    "Multi-head attention",
    "Feed-forward network",
    "Transformer block",
    "Layer normalization",
    "Residual connection",
    "Language-model output head"
  ],
  "common_module": "/content/drive/MyDrive/Scientific-Abstract-

Save Model Configuration

In [56]:
with open(
    MODEL_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        model_config.to_dict(),
        file,
        indent=2
    )

print(
    "Model configuration saved:"
)

print(
    MODEL_CONFIG_PATH
)

Model configuration saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/models/gpt_model_config.json


In [57]:
with open(
    MODEL_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        architecture_summary,
        file,
        indent=2
    )

print(
    "Model summary saved:"
)

print(
    MODEL_SUMMARY_PATH
)

Model summary saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/models/gpt_model_summary.json


Reload Saved Model Configuration

In [58]:
with open(
    MODEL_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:

    reloaded_config_dictionary = (
        json.load(file)
    )

reloaded_model_config = (
    GPTConfig.from_dict(
        reloaded_config_dictionary
    )
)

print(
    reloaded_model_config
)

GPTConfig(vocab_size=8000, block_size=256, n_embd=256, n_head=8, n_layer=6, dropout=0.1, bias=True)


In [59]:
configuration_reload_valid = (
    reloaded_model_config.to_dict()
    == model_config.to_dict()
)

print(
    "Configuration reload valid:",
    configuration_reload_valid
)

if not configuration_reload_valid:

    raise ValueError(
        "Reloaded model configuration "
        "does not match the original."
    )

Configuration reload valid: True


In [60]:
fresh_model = GPTLanguageModel(
    reloaded_model_config
).to(device)

fresh_model_parameter_count = (
    count_parameters(
        fresh_model
    )
)

print(
    "Fresh model created successfully."
)

print(
    "Fresh model parameter count:",
    f"{fresh_model_parameter_count:,}"
)

print(
    "Original model parameter count:",
    f"{trainable_parameters:,}"
)

Fresh model created successfully.
Fresh model parameter count: 6,852,608
Original model parameter count: 6,852,608


In [61]:
fresh_model_architecture_valid = (
    fresh_model_parameter_count
    == trainable_parameters
)

print(
    "Fresh-model architecture valid:",
    fresh_model_architecture_valid
)

if not fresh_model_architecture_valid:

    raise ValueError(
        "Fresh model does not match "
        "the original architecture."
    )

Fresh-model architecture valid: True


In [62]:
del attention_head
del causal_test_head
del multi_head_attention
del feed_forward
del transformer_block
del fresh_model

if torch.cuda.is_available():

    torch.cuda.empty_cache()

print(
    "Temporary test models removed."
)

Temporary test models removed.


In [63]:
if 'model' in locals() and model is not None:
    del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = GPTLanguageModel(model_config).to(device) # Re-initialize and move model to device for robust validation

validation_checks = {
    "common_module_available": (
        os.path.exists(
            COMMON_MODULE_PATH
        )
    ),
    "tokenizer_available": (
        os.path.exists(
            TOKENIZER_PATH
        )
    ),
    "tokenizer_config_available": (
        os.path.exists(
            TOKENIZER_CONFIG_PATH
        )
    ),
    "vocabulary_consistent": (
        int(saved_vocab_size)
        == vocab_size
    ),
    "special_tokens_available": (
        len(missing_tokens) == 0
    ),
    "model_created": (
        model is not None
    ),
    "model_on_selected_device": (
        model_on_selected_device
    ),
    "configuration_valid": (
        all(
            configuration_checks.values()
        )
    ),
    "causal_mask_valid": (
        causal_mask_is_valid
    ),
    "future_tokens_blocked": (
        causal_behavior_valid
    ),
    "forward_pass_valid": (
        actual_prompt_logits_shape
        == expected_prompt_logits_shape
    ),
    "loss_valid": (
        loss_is_scalar
        and loss_is_finite
        and loss_is_positive
    ),
    "gradients_finite": (
        gradient_values_are_finite
    ),
    "weights_tied": (
        weights_are_tied
    ),
    "maximum_context_supported": (
        maximum_context_logits.shape[1]
        == model_config.block_size
    ),
    "oversized_context_rejected": (
        oversized_context_rejected
    ),
    "model_config_saved": (
        os.path.exists(
            MODEL_CONFIG_PATH
        )
    ),
    "model_summary_saved": (
        os.path.exists(
            MODEL_SUMMARY_PATH
        )
    ),
    "configuration_reload_valid": (
        configuration_reload_valid
    ),
    "fresh_architecture_valid": (
        fresh_model_architecture_valid
    )
}

for check_name, result in (
    validation_checks.items()
):

    status = (
        "PASSED"
        if result
        else "FAILED"
    )

    print(
        f"{check_name}: {status}"
    )

common_module_available: PASSED
tokenizer_available: PASSED
tokenizer_config_available: PASSED
vocabulary_consistent: PASSED
special_tokens_available: PASSED
model_created: PASSED
model_on_selected_device: PASSED
configuration_valid: PASSED
causal_mask_valid: PASSED
future_tokens_blocked: PASSED
forward_pass_valid: PASSED
loss_valid: PASSED
gradients_finite: PASSED
weights_tied: PASSED
maximum_context_supported: PASSED
oversized_context_rejected: PASSED
model_config_saved: PASSED
model_summary_saved: PASSED
configuration_reload_valid: PASSED
fresh_architecture_valid: PASSED


Completion Message

In [64]:
if all(
    validation_checks.values()
):

    print("=" * 80)

    print(
        "03_GPT_Model.ipynb "
        "completed successfully."
    )

    print("=" * 80)

    print(
        "\nModel type:"
    )

    print(
        "Decoder-only GPT Transformer"
    )

    print(
        "\nVocabulary size:"
    )

    print(
        f"{model_config.vocab_size:,}"
    )

    print(
        "\nContext length:"
    )

    print(
        model_config.block_size
    )

    print(
        "\nEmbedding dimension:"
    )

    print(
        model_config.n_embd
    )

    print(
        "\nAttention heads:"
    )

    print(
        model_config.n_head
    )

    print(
        "\nTransformer layers:"
    )

    print(
        model_config.n_layer
    )

    print(
        "\nTrainable parameters:"
    )

    print(
        f"{trainable_parameters:,}"
    )

    print(
        "\nModel configuration saved at:"
    )

    print(
        MODEL_CONFIG_PATH
    )

    print(
        "\nNext notebook:"
    )

    print(
        "04_Training.ipynb"
    )

else:

    failed_checks = [
        check_name
        for check_name, result
        in validation_checks.items()
        if not result
    ]

    raise RuntimeError(
        "GPT model validation failed: "
        f"{failed_checks}"
    )

03_GPT_Model.ipynb completed successfully.

Model type:
Decoder-only GPT Transformer

Vocabulary size:
8,000

Context length:
256

Embedding dimension:
256

Attention heads:
8

Transformer layers:
6

Trainable parameters:
6,852,608

Model configuration saved at:
/content/drive/MyDrive/Scientific-Abstract-GPT/models/gpt_model_config.json

Next notebook:
04_Training.ipynb
